In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable

In [0]:
BRONZE_PATH = "abfss://bronze@pravdatalake.dfs.core.windows.net"
SILVER_PATH = "abfss://silver@pravdatalake.dfs.core.windows.net"
SILVER_TABLE_PATH = f"{SILVER_PATH}/trim_table"
SILVER_TABLE_NAME = "vehicle_sales.silver.trim_table"

In [0]:
trim_df = spark.read.format("csv")\
    .option("header", True)\
    .option("inferSchema", True)\
    .load(f"{BRONZE_PATH}/Trim_table")


In [0]:
trim_df.display()

In [0]:
print(f"bronze row count: {trim_df.count()}")

In [0]:
silver_trim = (
    trim_df
    .withColumn("Genmodel_ID", trim(col("Genmodel_ID")))
    .withColumn("Maker", trim(initcap(col("Maker"))))
    .withColumn("Genmodel", trim(col("Genmodel")))
    .withColumn("Trim", trim(col("Trim")))
    .withColumn("Year", col("Year").cast(IntegerType()))
    .withColumn("Price", col("Price").cast(DoubleType()))
    .withColumn("Gas_emission", col("Gas_emission").cast(DoubleType()))
    .withColumn("Fuel_type", trim(initcap(col("Fuel_type"))))
    .withColumn("Engine_size", col("Engine_size").cast(DoubleType()))
    .filter(col("Genmodel_ID").isNotNull())
    .filter(col("Trim").isNotNull())
    .filter(col("Year").isNotNull())
    .filter((col("Price").isNull()) | (col("Price") >= 0))
    .dropDuplicates(["Genmodel_ID", "Trim", "Year"])
    .withColumn("silver_ingestion_timestamp", current_timestamp())
)

In [0]:
silver_trim.display()

####Data Quality Checks

In [0]:
row_count = silver_trim.count()

In [0]:
null_key_count = silver_trim.filter(col("Genmodel_ID").isNull() | col("Trim").isNull() | col("Year").isNull()).count()

In [0]:
duplicate_key_count = silver_trim.groupBy("Genmodel_ID", "Trim", "Year").count().filter("count > 1").count()

In [0]:
negative_price_count = silver_trim.filter(col("Price") < 0).count()

In [0]:
print(f"silver row count: {row_count}")
print(f"null key count: {null_key_count}")
print(f"duplicate key count: {duplicate_key_count}")
print(f"negative Price count: {negative_price_count}")

In [0]:
assert null_key_count == 0, "Genmodel_ID/Trim/Year should never be null in silver_trim"
assert duplicate_key_count == 0, "(Genmodel_ID, Trim, Year) should be unique in silver_trim"
assert negative_price_count == 0, "Price should never be negative"

In [0]:
if DeltaTable.isDeltaTable(spark, SILVER_TABLE_PATH):
 
    silver_table = DeltaTable.forPath(spark, SILVER_TABLE_PATH)
 
    (silver_table.alias("t")
        .merge(
            silver_trim.alias("s"),
            "t.Genmodel_ID = s.Genmodel_ID AND t.Trim = s.Trim AND t.Year = s.Year"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
 
else:
 
    silver_trim.write \
        .format("delta") \
        .mode("overwrite") \
        .partitionBy("Year") \
        .save(SILVER_TABLE_PATH)

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {SILVER_TABLE_NAME}
    USING DELTA
    LOCATION '{SILVER_TABLE_PATH}'
""")

In [0]:
spark.sql(f"OPTIMIZE {SILVER_TABLE_NAME} ZORDER BY (Genmodel_ID)")